In [3]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ─── Configuration ─────────────────────────────────────────────────────────────
data_folder = r"D:\datasets\pcaps_cic\flow_features"
benign_csv  = os.path.join(data_folder, "Benign_flows.csv")
ddos_csv    = os.path.join(data_folder, "DDoS_flows.csv")

# Load & label
df_b = pd.read_csv(benign_csv); df_b["label"] = 0
df_d = pd.read_csv(ddos_csv);   df_d["label"] = 1
df  = pd.concat([df_b, df_d], ignore_index=True)

# 1) Filter down to N = 8
df = df[df["N"] == 8].copy()

# 2) Select only the 8 features + the label
selected_feats = [
    "iat_mean",
    "iat_min",
    "iat_max",
    "ip_len_mean",
    "ip_len_min",
    "ip_len_max",
    "tcp_flag_count",
    "bytes_sent"
]
df = df[selected_feats + ["label"]]

# 3) Sanity check: no extra cols
extra = set(df.columns) - set(selected_feats) - {"label"}
assert not extra, f"Unexpected columns in df: {extra}"

print("Columns going into training:", df.columns.tolist())

# 4) Split & scale
X = df[selected_feats].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

scaler    = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Ready to train on features:", selected_feats)


Columns going into training: ['iat_mean', 'iat_min', 'iat_max', 'ip_len_mean', 'ip_len_min', 'ip_len_max', 'tcp_flag_count', 'bytes_sent', 'label']
Ready to train on features: ['iat_mean', 'iat_min', 'iat_max', 'ip_len_mean', 'ip_len_min', 'ip_len_max', 'tcp_flag_count', 'bytes_sent']


In [ ]:

import torch
import torch.nn as nn
import numpy as np

output_folder = r"./mlp_headers_demo"
os.makedirs(output_folder, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}  |  PyTorch {torch.__version__}")

# ── MLP (mirrors sklearn MLPClassifier with relu activations) ────────────────
class MLP(nn.Module):
    def __init__(self, in_features, hidden_sizes, n_classes=2):
        super().__init__()
        layers, prev = [], in_features
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

layer_specs = [
    (8,),
    (32,),
    (64, 32),
    (128, 64, 32),
    (256, 128, 64, 32),
]

# ── C header export (compatible with existing scalar / NEON inference) ────────
def write_array_to_c(name, array, f):
    flat = array.flatten()
    f.write(f"// Shape: {array.shape}\n")
    f.write(f"static const float {name}[{len(flat)}] ALIGN16 = {{\n    ")
    f.write(", ".join(f"{x:.6f}f" for x in flat))
    f.write("\n};\n\n")

def export_mlp_to_c(model, out_path):
    linears = [m for m in model.net if isinstance(m, nn.Linear)]
    n_layers = len(linears)
    with open(out_path, "w") as f:
        f.write("// Auto-generated MLP weights for C inference\n\n")
        f.write("#pragma once\n\n")
        f.write("#define ALIGN16 __attribute__((aligned(16)))\n\n")
        f.write(f"#define NUM_LAYERS {n_layers}\n\n")
        sizes = [linears[0].in_features] + [l.out_features for l in linears]
        f.write("static const int LAYER_SIZES[NUM_LAYERS+1] = { " +
                ", ".join(str(s) for s in sizes) + " };\n\n")
        for idx, layer in enumerate(linears):
            # PyTorch weight is (out, in) — transpose to (in, out) for C convention
            W = layer.weight.detach().cpu().numpy().T
            b = layer.bias.detach().cpu().numpy()
            write_array_to_c(f"W{idx}", W, f)
            write_array_to_c(f"B{idx}", b, f)
        f.write("static const float *WEIGHTS[NUM_LAYERS] = { " +
                ", ".join(f"W{j}" for j in range(n_layers)) + " };\n")
        f.write("static const float *BIASES[NUM_LAYERS]  = { " +
                ", ".join(f"B{j}" for j in range(n_layers)) + " };\n\n")
        f.write("#undef ALIGN16\n")


In [ ]:

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report

EPOCHS   = 30
BATCH_SZ = 256
LR       = 1e-3
N_IN     = X_train_s.shape[1]   # 8

X_tr_t = torch.tensor(X_train_s, dtype=torch.float32)
y_tr_t = torch.tensor(y_train,   dtype=torch.long)
X_te_t = torch.tensor(X_test_s,  dtype=torch.float32).to(device)

train_ds = TensorDataset(X_tr_t, y_tr_t)
loader   = DataLoader(train_ds, batch_size=BATCH_SZ, shuffle=True,
                      num_workers=0, pin_memory=(device.type == "cuda"))

models = {}   # kept for ONNX export

for spec in layer_specs:
    name      = "_".join(str(x) for x in spec)
    model_var = f"mlp_{name}"
    print(f"Training {model_var}  layers={spec}…")

    model   = MLP(N_IN, spec).to(device)
    opt     = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(EPOCHS):
        running = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            running += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"  epoch {epoch+1:3d}/{EPOCHS}  loss={running/len(loader):.4f}")

    model.eval()
    with torch.no_grad():
        y_pred = model(X_te_t).argmax(dim=1).cpu().numpy()

    print(f"Classification report for {model_var}:")
    print(classification_report(y_test, y_pred, target_names=["Benign", "DDoS"]))

    models[name] = model.cpu().eval()

    header_path = os.path.join(output_folder, f"{model_var}.h")
    export_mlp_to_c(models[name], header_path)
    print(f"  → Wrote {header_path}\n")


In [9]:
means = scaler.mean_    # numpy array, shape (n_features,)
stds  = scaler.scale_   # numpy array, shape (n_features,)

# Print them out
print("FEATURE_MEAN = [")
for m in means:
    print(f"    {m:.6f}f,")
print("]\n")

print("FEATURE_STD = [")
for s in stds:
    print(f"    {s:.6f}f,")
print("]")


FEATURE_MEAN = [
    37.389100f,
    0.000000f,
    152.635795f,
    66.589368f,
    42.522597f,
    152.613895f,
    7.816264f,
    532.714945f,
]

FEATURE_STD = [
    576.053449f,
    1.000000f,
    4608.169432f,
    113.323397f,
    15.743588f,
    441.474377f,
    1.198385f,
    906.587174f,
]


In [10]:
header_path = "feature_stats.h"
with open(header_path, "w") as f:
    f.write("// Auto-generated feature stats for z-score normalization\n\n")
    f.write("#pragma once\n\n")
    f.write("#define ALIGN16 __attribute__((aligned(16)))\n")
    f.write(f"#define NUM_FEATURES {len(means)}\n\n")
    # Means
    f.write("static const float FEATURE_MEAN[NUM_FEATURES] ALIGN16 = {\n    ")
    f.write(", ".join(f"{m:.6f}f" for m in means))
    f.write("\n};\n\n")
    # Stds
    f.write("static const float FEATURE_STD[NUM_FEATURES] ALIGN16 = {\n    ")
    f.write(", ".join(f"{s:.6f}f" for s in stds))
    f.write("\n};\n\n")
    f.write("#undef ALIGN16\n")


## ONNX Export & ORT Inference (BF3 / DPDK target)

Export all trained sklearn MLP variants to ONNX, validate numerical equivalence,
benchmark ORT vs sklearn throughput across batch sizes, then generate
`mlp_ort_dpdk.h` — a drop-in header for DPDK multicore lcore workers.

**Pipeline on BF3:**
```
rte_eth_rx_burst → extract features → z-score norm (feature_stats.h)
    → mlp_ort_infer()  (ORT, 1 session / lcore)
    → drop DDoS / forward Benign
```
Compare against the ARM NEON scalar C path from the existing `.h` headers.

In [ ]:

# ─── Models stored during training — verify before ONNX export ───────────────
print(f"{'Model':<25}  {'Architecture':>30}  {'Params':>8}")
print("-" * 68)
for name, m in models.items():
    linears = [l for l in m.net if isinstance(l, nn.Linear)]
    arch    = "→".join(str(l.in_features) for l in linears) + f"→{linears[-1].out_features}"
    params  = sum(p.numel() for p in m.parameters())
    print(f"  mlp_{name:<21}  {arch:>30}  {params:>8,}")


In [ ]:

# ─── Export PyTorch MLPs → ONNX via torch.onnx.export ────────────────────────
onnx_folder = r"./mlp_onnx_models"
os.makedirs(onnx_folder, exist_ok=True)

N_FEAT     = X_train_s.shape[1]   # 8
onnx_paths = {}
dummy      = torch.zeros(1, N_FEAT)   # batch dim is dynamic

print(f"{'Model':<22}  {'Size (KB)':>10}  {'Path'}")
print("-" * 68)

for name, model in models.items():
    model.eval()
    out_path = os.path.join(onnx_folder, f"mlp_{name}.onnx")
    with torch.no_grad():
        torch.onnx.export(
            model, dummy, out_path,
            input_names   = ["float_input"],
            output_names  = ["logits"],
            dynamic_axes  = {"float_input": {0: "batch_size"},
                             "logits":      {0: "batch_size"}},
            opset_version = 17,
        )
    onnx_paths[name] = out_path
    kb = os.path.getsize(out_path) / 1024
    print(f"  mlp_{name:<18}  {kb:>9.1f}  {out_path}")

print(f"\nAll {len(onnx_paths)} models exported.")
print("Output tensor: 'logits'  float32 [batch, 2]  —  argmax → class label")
print("DPDK target  : mlp_128_64_32")


In [ ]:

# ─── Validate: ORT logits must match PyTorch logits numerically ───────────────
import onnxruntime as ort
import numpy as np

print(f"ORT version : {ort.__version__}\n")
print(f"{'Model':<22}  {'Label match / 1000':>20}  {'Max logit Δ':>12}")
print("-" * 60)

X_np = X_test_s[:1000].astype(np.float32)
X_pt = torch.tensor(X_np)

for name, path in onnx_paths.items():
    sess     = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    inp_name = sess.get_inputs()[0].name

    ort_logits = sess.run(None, {inp_name: X_np})[0]          # (1000, 2) float32
    ort_labels = np.argmax(ort_logits, axis=1)

    with torch.no_grad():
        pt_logits  = models[name](X_pt).numpy()               # (1000, 2)
    pt_labels  = np.argmax(pt_logits, axis=1)

    match     = int(np.sum(ort_labels == pt_labels))
    max_delta = float(np.max(np.abs(ort_logits - pt_logits)))
    print(f"  mlp_{name:<18}  {match:>8}/1000             {max_delta:>10.2e}")


In [ ]:

# ─── Benchmark: PyTorch CPU vs ORT across batch sizes ─────────────────────────
import time

TARGET  = "128_64_32"
model   = models[TARGET].eval()
sess    = ort.InferenceSession(onnx_paths[TARGET],
                               providers=["CPUExecutionProvider"])
inp     = sess.get_inputs()[0].name
REPEATS = 300

print(f"Model: mlp_{TARGET}")
print(f"{'Mode':<18} {'Batch':>6}  {'ms/batch':>10}  {'Mflows/s':>10}  {'µs/flow':>9}")
print("-" * 62)

for BATCH in [1, 8, 32, 64, 128, 256, 512, 1024]:
    X_b_np = X_test_s[:BATCH].astype(np.float32)
    X_b_pt = torch.tensor(X_b_np)

    # PyTorch CPU
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(REPEATS):
            model(X_b_pt).argmax(dim=1)
    pt_ms = (time.perf_counter() - t0) / REPEATS * 1000

    # ORT
    t0 = time.perf_counter()
    for _ in range(REPEATS):
        sess.run(None, {inp: X_b_np})
    ort_ms = (time.perf_counter() - t0) / REPEATS * 1000

    print(f"  {'PyTorch CPU':<16} {BATCH:>6}  {pt_ms:>10.3f}  {BATCH/pt_ms*1e-3:>10.2f}  {pt_ms*1000/BATCH:>8.1f}")
    print(f"  {'ORT (ONNX)':<16} {BATCH:>6}  {ort_ms:>10.3f}  {BATCH/ort_ms*1e-3:>10.2f}  {ort_ms*1000/BATCH:>8.1f}  ×{pt_ms/ort_ms:.1f}")
    print()


In [ ]:

# ─── Generate DPDK multicore ORT integration header ───────────────────────────
TARGET_ONNX = os.path.abspath(onnx_paths["128_64_32"])
N_OUT       = 2
MLP_MAX_BATCH = 64    # max DPDK RX burst — adjust to match rte_eth_rx_burst limit

dpdk_ort_h = r"./mlp_ort_dpdk.h"

code = f"""\
/* Auto-generated — DPDK multicore ORT inference integration
 * Model  : mlp_128_64_32  (8→128→64→32→2)
 * ONNX   : {TARGET_ONNX}
 *
 * Each DPDK lcore owns its own OrtSession (no locking needed).
 * Flow features must be z-score normalised (feature_stats.h) before calling
 * mlp_ort_infer().  Output is raw logits — argmax done in C, no alloc.
 *
 * Compile flags:
 *   -I$(ORT_ROOT)/include
 *   -L$(ORT_ROOT)/lib -lonnxruntime
 */
#pragma once

#include <stdint.h>
#include <stdio.h>
#include <onnxruntime_c_api.h>
#include <rte_lcore.h>
#include <rte_log.h>

#define MLP_N_FEAT    {N_FEAT}
#define MLP_N_OUT     {N_OUT}
#define MLP_MAX_BATCH {MLP_MAX_BATCH}
#define MLP_ONNX_PATH "{TARGET_ONNX.replace(chr(92), chr(92)+chr(92))}"

/* ── Per-lcore context ───────────────────────────────────────────────────── */
typedef struct {{
    const OrtApi  *api;
    OrtEnv        *env;
    OrtSession    *session;
    OrtMemoryInfo *mem_info;
    /* static logit buffer — avoids heap alloc in hot path */
    float          logit_buf[MLP_MAX_BATCH * MLP_N_OUT];
}} mlp_ort_ctx_t;

static mlp_ort_ctx_t g_ort_ctx[RTE_MAX_LCORE];

/* ── Initialise ORT session — call once per lcore at startup ─────────────── */
static inline int
mlp_ort_lcore_init(void)
{{
    unsigned       lcore = rte_lcore_id();
    mlp_ort_ctx_t *ctx   = &g_ort_ctx[lcore];
    OrtStatus     *st;

    ctx->api = OrtGetApiBase()->GetApi(ORT_API_VERSION);
    if (!ctx->api) {{
        RTE_LOG(ERR, USER1, "ORT: GetApi failed lcore %u\\n", lcore);
        return -1;
    }}

    st = ctx->api->CreateEnv(ORT_LOGGING_LEVEL_WARNING, "mlp_dpdk", &ctx->env);
    if (st) {{ ctx->api->ReleaseStatus(st); return -1; }}

    OrtSessionOptions *opts;
    st = ctx->api->CreateSessionOptions(&opts);
    if (st) {{ ctx->api->ReleaseStatus(st); return -1; }}
    ctx->api->SetIntraOpNumThreads(opts, 1);
    ctx->api->SetSessionGraphOptimizationLevel(opts, ORT_ENABLE_ALL);

    st = ctx->api->CreateSession(ctx->env, MLP_ONNX_PATH, opts, &ctx->session);
    ctx->api->ReleaseSessionOptions(opts);
    if (st) {{
        RTE_LOG(ERR, USER1, "ORT: CreateSession failed lcore %u\\n", lcore);
        ctx->api->ReleaseStatus(st); return -1;
    }}

    st = ctx->api->CreateCpuMemoryInfo(OrtArenaAllocator, OrtMemTypeDefault,
                                       &ctx->mem_info);
    if (st) {{ ctx->api->ReleaseStatus(st); return -1; }}

    RTE_LOG(INFO, USER1, "ORT: lcore %u ready\\n", lcore);
    return 0;
}}

/* ── Run inference on a burst of flows ───────────────────────────────────── *
 * feats      float[batch_size * MLP_N_FEAT]  z-score normalised, row-major
 * preds      int32_t[batch_size]             0=Benign  1=DDoS
 * returns 0 on success, -1 on error
 *
 * No heap allocation in hot path — logits land in ctx->logit_buf.
 */
static inline int
mlp_ort_infer(const float *feats, int32_t *preds, int batch_size)
{{
    unsigned       lcore  = rte_lcore_id();
    mlp_ort_ctx_t *ctx    = &g_ort_ctx[lcore];
    const OrtApi  *api    = ctx->api;
    OrtStatus     *st;

    /* input tensor — zero-copy wrap of caller's buffer */
    int64_t  in_shape[2] = {{batch_size, MLP_N_FEAT}};
    OrtValue *in_val     = NULL;
    st = api->CreateTensorWithDataAsOrtValue(
            ctx->mem_info, (void *)feats,
            (size_t)batch_size * MLP_N_FEAT * sizeof(float),
            in_shape, 2, ONNX_TENSOR_ELEMENT_DATA_TYPE_FLOAT, &in_val);
    if (st) {{ api->ReleaseStatus(st); return -1; }}

    /* output tensor — zero-copy wrap of static logit buffer */
    int64_t  out_shape[2] = {{batch_size, MLP_N_OUT}};
    OrtValue *out_val     = NULL;
    st = api->CreateTensorWithDataAsOrtValue(
            ctx->mem_info, (void *)ctx->logit_buf,
            (size_t)batch_size * MLP_N_OUT * sizeof(float),
            out_shape, 2, ONNX_TENSOR_ELEMENT_DATA_TYPE_FLOAT, &out_val);
    if (st) {{ api->ReleaseValue(in_val); api->ReleaseStatus(st); return -1; }}

    const char *in_names[]  = {{"float_input"}};
    const char *out_names[] = {{"logits"}};

    st = api->Run(ctx->session, NULL,
                  in_names,  (const OrtValue *const *)&in_val,  1,
                  out_names, 1, &out_val);
    api->ReleaseValue(in_val);
    api->ReleaseValue(out_val);
    if (st) {{ api->ReleaseStatus(st); return -1; }}

    /* argmax(logits[i,0], logits[i,1]) — no SIMD needed, dominates nothing */
    for (int i = 0; i < batch_size; i++)
        preds[i] = (ctx->logit_buf[i * MLP_N_OUT + 1] >
                    ctx->logit_buf[i * MLP_N_OUT + 0]) ? 1 : 0;

    return 0;
}}

/* ── Release per-lcore ORT resources ─────────────────────────────────────── */
static inline void
mlp_ort_lcore_cleanup(void)
{{
    unsigned       lcore = rte_lcore_id();
    mlp_ort_ctx_t *ctx   = &g_ort_ctx[lcore];
    if (ctx->api) {{
        ctx->api->ReleaseMemoryInfo(ctx->mem_info);
        ctx->api->ReleaseSession(ctx->session);
        ctx->api->ReleaseEnv(ctx->env);
    }}
}}
"""

with open(dpdk_ort_h, "w") as f:
    f.write(code)

print(f"Written : {dpdk_ort_h}")
print()
print("DPDK lcore loop pattern:")
print("  static int lcore_worker(void *arg) {")
print("      mlp_ort_lcore_init();")
print("      while (running) {")
print("          n = rte_eth_rx_burst(..., pkts, MLP_MAX_BATCH);")
print("          extract_features(pkts, feats, n);")
print("          normalize(feats, n, FEATURE_MEAN, FEATURE_STD); // feature_stats.h")
print("          mlp_ort_infer(feats, preds, n);                  // 0=benign 1=DDoS")
print("          dispatch(pkts, preds, n);")
print("      }")
print("      mlp_ort_lcore_cleanup();")
print("  }")
